In [6]:
import numpy as np
import time

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report,confusion_matrix
import warnings
warnings.filterwarnings('ignore',category=FutureWarning)


In [11]:
print("Downloading Fashion MNIST dataset(this may take 30-60 seconds..")

fashion_mnist=fetch_openml('Fashion-MNIST',version=1,as_frame=False)

x=fashion_mnist.data
y=fashion_mnist.target.astype(int)

print(f"Total dataset size:{x.shape[0]} images, each with {x.shape[1]} pixels.")

Total dataset size:70000 images, each with 784 pixels.


In [29]:
_,x_subset,_,y_subset=train_test_split(
    x,y,
    test_size=12000,
    stratify=y,
    random_state=42
)
x_train,x_test,y_train,y_test=train_test_split(
    x_subset,y_subset,
    test_size=2000,
    stratify=y_subset,
    random_state= 42
)

print(f"Training images: {x_train.shape[0]}")
print(f"Testing images: {x_test.shape[0]}")

Training images: 10000
Testing images: 2000


In [30]:
print(f"Before Scaling->Min: {x_train.min()},Max: {x_train.max()}")

x_train=x_train/255.0
x_test=x_test/255.0

print(f"After Scaling->Min: {x_train.min()},Max: {x_train.max()}")

Before Scaling->Min: 0,Max: 255
After Scaling->Min: 0.0,Max: 1.0


In [31]:
k_values=[1,3,5,7,9,15]
results={}
print(f"{'K Value':<8}|{'Accuracy':<10}|{'Prediction Time(seconds)':<25}")
print("-"*50)

for k in k_values:
    knn=KNeighborsClassifier(n_neighbors=k,metric='euclidean',n_jobs=-1)
    knn.fit(x_train,y_train)
    start_time=time.time()
    y_pred=knn.predict(x_test)
    elapsed_time=time.time()-start_time

    acc=accuracy_score(y_test,y_pred)

    results[k]={
        "accuracy":acc,
        "time":elapsed_time,
        "predictions":y_pred
    }

    print(f"{k:<8} | {acc *100:<9.2f}%| {elapsed_time:<25.2f}")

K Value |Accuracy  |Prediction Time(seconds) 
--------------------------------------------------
1        | 80.50    %| 2.91                     
3        | 82.40    %| 0.20                     
5        | 82.90    %| 0.21                     
7        | 81.95    %| 0.21                     
9        | 82.05    %| 0.21                     
15       | 81.60    %| 0.21                     


In [34]:
class_names=[
"T-shirt/top","Trouser","Pullover","Dress","Coat",
"Sandal","Shirt","Sneaker","Bag","Ankle Boot"
]
best_k=max(results,key=lambda k:results[k]["accuracy"])
print(f"Best K is:{best_k} with {results[best_k]['accuracy']*100:.2f}%accuracy\n")
print("Per-Class Classification Report:")
print(classification_report(y_test,results[best_k]["predictions"],target_names=class_names))

Best K is:5 with 82.90%accuracy

Per-Class Classification Report:
              precision    recall  f1-score   support

 T-shirt/top       0.75      0.86      0.80       200
     Trouser       0.97      0.94      0.96       200
    Pullover       0.68      0.78      0.73       200
       Dress       0.87      0.83      0.85       200
        Coat       0.73      0.74      0.73       200
      Sandal       0.99      0.83      0.90       200
       Shirt       0.62      0.51      0.55       200
     Sneaker       0.85      0.92      0.88       200
         Bag       0.98      0.92      0.95       200
  Ankle Boot       0.88      0.96      0.92       200

    accuracy                           0.83      2000
   macro avg       0.83      0.83      0.83      2000
weighted avg       0.83      0.83      0.83      2000

